#Transform Drivers Data
1. Read bronze drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (driverId → driver_id.dateOfbirth →date_of_birth )
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and  transform the value to Title Case
5. Remove duplicate records
6. Transform column values of nationality to Title Case
7. Write the transformed data to silver drivers table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")  

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"


Step 1-Read bronze drivers table

In [0]:
from pyspark.sql import functions as F
drivers_df = (
    spark.table(bronze_table)
         .filter((F.col("batch_id") == v_batch_id))
)
#                            (OR)
# drivers_df = spark.read.option("versionAsOf",0).table(bronze_table)

In [0]:
display(drivers_df)

Step 2- Keep only the columns required for analytics (Drop url column)

In [0]:
drivers_dropped_df = drivers_df.drop(F.col("url"))


In [0]:
display(drivers_dropped_df)

Step 3:
- Standardize column names using snake_case (driverId → driver_id.dateOfbirth → date_of_birth)

In [0]:
"""circuits_renamed_df = (

        circuits_selected_df
            .withColumnRenamed("circuitId","circuit_id")
            .withColumnRenamed("raceName","race_name")
            .withColumnRenamed("date","race_date") 
                                                    )

                      OR                             """
drivers_renamed_df = (drivers_dropped_df
                       .withColumnsRenamed({
                           "driverId":"driver_id",
                           "dateOfBirth":"date_of_birth"})
                       )


display(drivers_renamed_df)




Step 4- Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case


In [0]:
drivers_concatenated_df = (drivers_renamed_df
                                .withColumn("driver_name",
                                           F.initcap(F.concat_ws(" ",F.col("name.givenName"),F.col("name.familyName"))))
                                .drop("name")
                           
                           )
display(drivers_concatenated_df)


Step 5- Remove duplicate records

In [0]:
"""constructors_distinct_df = constructors_renamed_df.distinct()
                        OR  """
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])
drivers_distinct_df.display()

Step 6- Transform column values of nationality to Title Case

In [0]:
drivers_final_df = (drivers_distinct_df
                            .withColumn("nationality",F.initcap(F.col("nationality")))

)
drivers_final_df.display()



Step 8- Write the transformed data to silver drivers table

In [0]:
write_to_silver(
    input_df=drivers_final_df,
    target_table=silver_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))